# 13 · Korean PDF chunk quality

Why the corpus_v2 guidebook *한국 유학생 생활 가이드북* (94 pages, 291 chunks, median 37 characters) is fragmented, and whether better extraction / chunking changes retrieval. Each variant replaces only the guidebook text in an otherwise identical corpus_v2, is indexed with the production chunker and embedding model in a separate local in-memory index, and is evaluated on the same frozen questions. Production data and embeddings are not touched.

**Inputs** are versioned files under `research/data/` and `research/configs/`; nothing here connects to the production database or writes to it. Run top to bottom (Kernel → Restart & Run All).

In [ ]:
import sys, json, platform, warnings
from pathlib import Path
import numpy as np, pandas as pd, scipy, matplotlib
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

RESEARCH = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd() / "research"
sys.path.insert(0, str(RESEARCH / "scripts"))
import analysis as A

A.style()
np.random.seed(A.SEED)
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 30); pd.set_option("display.width", 160)
EXPERIMENTS = json.loads((RESEARCH / "configs/experiments.json").read_text())
print(f"python {platform.python_version()} | pandas {pd.__version__} | numpy {np.__version__} | "
      f"scipy {scipy.__version__} | matplotlib {matplotlib.__version__} | seed {A.SEED}")
print(f"dataset snapshot: {A.SNAPSHOT} (set RAG_SNAPSHOT=corpus_v1|corpus_v2) | results: {A.RESULTS.relative_to(RESEARCH)} | "
      f"figures: {A.FIGURES.relative_to(RESEARCH)}")

In [ ]:
import csv, re
EXP = A.DATA / "experiments/chunking"; RES = EXP / "results"; FIG = A.RESEARCH / "figures/chunk_quality"
REP = A.RESEARCH / "reports/chunk_quality"; REP.mkdir(parents=True, exist_ok=True)
INV = json.loads((EXP / "investigation.json").read_text())
VM = json.loads((EXP / "variants.manifest.json").read_text())
VARIANTS = ["C0_existing", "C1_merge_short", "C2_paragraph", "C3_heading", "C4_table", "P1_prod_fixed_body"]
LABEL = {"C0_existing": "C0 existing (production)", "C1_merge_short": "C1 merge short fragments",
         "C2_paragraph": "C2 paragraph-aware", "C3_heading": "C3 heading-aware", "C4_table": "C4 table-aware",
         "P1_prod_fixed_body": "P1 production extractor, fixed body size", "C0_repeat": "C0 rebuilt (noise floor)"}
GB = "d60bc363-0aae-44e5-b0c3-df8c61ad6f66"
BENCH = {ds: {r["question_id"]: r for r in json.loads((A.DATA / f"benchmarks/retrieval_{ds}.json").read_text())["records"]}
         for ds in ("pilot_v0.1", "heldout_v0.2")}
QDS = {q: ds for ds, recs in BENCH.items() for q in recs}
REC = {q: r for recs in BENCH.values() for q, r in recs.items()}
FROZEN = {q: set(r["expected_document_ids"]) for q, r in REC.items()}
SENS = json.loads((EXP / "label_sensitivity_ai_proposed.json").read_text())
AI = {q: FROZEN[q] | set(SENS["add"].get(r["scenario_id"], [])) for q, r in REC.items()}
man = {i["id"]: i for i in json.loads((A.DATA / "snapshots/corpus_v2/corpus_manifest.json").read_text())["items"]}
CROSS = {q: r["language"] != man[r["primary_document_id"]]["language"] for q, r in REC.items()}
RUNS = [v for v in VARIANTS + ["C0_repeat"] if (RES / f"retrieval_{v}.csv").exists()]
def load(v, labels):
    df = A.score_rankings(pd.read_csv(RES / f"retrieval_{v}.csv"), labels)
    return df.assign(variant=v, dataset=df.question_id.map(QDS), cross_language=df.question_id.map(CROSS))
RET = {lab: pd.concat([load(v, L) for v in RUNS], ignore_index=True) for lab, L in (("frozen", FROZEN), ("ai_sensitivity", AI))}
CH = {v: pd.read_csv(RES / f"chunks_{v}.csv") for v in VARIANTS if (RES / f"chunks_{v}.csv").exists()}
print("variants evaluated:", RUNS)
print("label sets: frozen v0.1/v0.2 (official) and", SENS["status"], f"({sum(len(x) for x in SENS['add'].values())} added credits)")

## 1 · Dataset statistics and extraction diagnosis

The production text was extracted by PyMuPDF (`table_mode=off`). The diagnostics separate the candidate causes: extraction (heading tagging, line-level paragraphs), Korean sentence boundaries, tables / layout, headers and footers, OCR / encoding, the chunker and empty paragraphs.

In [ ]:
f, x, l, c = INV["font"], INV["extracted_text"], INV["layout"], INV["chunker"]
pm = INV["production_extraction_meta"]
rows = [
    ("PDF pages / pages with text / image-only pages", f"{pm['pages']} / {pm['pages_with_text']} / {pm['empty_pages']}"),
    ("OCR used", "no (3 image-only pages were skipped and flagged)"),
    ("Body font size estimated by production (first 20 pages)", f"{f['body_estimate_production_first20']} pt"),
    ("Body font size over all pages (character-weighted mode)", f"{f['body_estimate_all_pages']} pt"),
    ("Heading threshold used (1.2 x estimate)", f"{f['heading_threshold_pt']} pt"),
    ("Share of all characters printed at >= threshold -> tagged '##'", f"{f['share_chars_tagged_heading']:.0%}"),
    ("Extracted paragraphs / marked as headings", f"{x['paragraphs']} / {x['heading_marked']}"),
    ("'Headings' longer than 40 characters (body sentences)", x["heading_marked_over_40_chars"]),
    ("Median paragraph length (characters)", x["median_paragraph_chars"]),
    ("Paragraphs ending a Korean sentence (다/요/. ...)", f"{x['paragraphs_ending_sentence']} of {x['paragraphs']}"),
    ("Paragraphs < 20 characters", x["paragraphs_under_20_chars"]),
    ("Page-number-only paragraphs / running header / chapter tabs", f"{x['page_number_only']} / {x['running_header']} / {x['chapter_tabs']}"),
    ("Pages with side-by-side layout", l["two_column_like_pages"]),
    ("Tables detected by PyMuPDF (extraction had tables off)", VM["c4_tables"]["kept"] + VM["c4_tables"]["rejected"]),
    ("Control / zero-width characters in the PDF text", ", ".join(f"{k} x{v}" for k, v in l["control_or_zero_width_chars"].items())),
    ("Replacement characters (encoding errors)", x["replacement_chars"]),
    ("Chunker: chunks / boundaries forced by a heading change", f"{c['chunks']} / {c['chunk_boundaries_at_heading_change']} ({c['share_boundaries_forced_by_heading']:.0%})"),
]
display(pd.DataFrame(rows, columns=["diagnostic", "value"]))

In [ ]:
cov = []
import sys as _s; _s.path.insert(0, str(A.RESEARCH.parent / "backend"))
from app.services.chunking import chunk_text, ChunkerConfig
import collections
for v in VARIANTS:
    t = (EXP / v / "guidebook.md").read_text()
    ch = chunk_text(t, ChunkerConfig(320, 420, 48))
    src = collections.Counter(re.findall(r"[가-힣]+", re.sub(r"\[Page \d+\]", "", t)))
    con = collections.Counter(w for k in ch for w in re.findall(r"[가-힣]+", k.content))
    both = con + collections.Counter(w for k in ch for w in re.findall(r"[가-힣]+", k.heading or ""))
    share = lambda C: sum(min(n, C[w]) for w, n in src.items()) / sum(src.values())
    cov.append(dict(variant=LABEL[v], hangul_words=sum(src.values()), in_chunk_text=share(con), in_text_or_heading=share(both)))
cov = pd.DataFrame(cov); display(cov.round(3))
c0 = cov.iloc[0]
display(Markdown(f"""**Diagnosis.** The fragmentation starts in *PDF text extraction*, and the chunker turns it into
*content loss*:

1. The production extractor estimates the body font size from the first 20 pages only. Those pages are front matter
   and visa forms printed at {f['body_estimate_production_first20']} pt, while the body text is {f['body_estimate_all_pages']} pt.
   Every short line at >= {f['heading_threshold_pt']} pt becomes a `##` heading: {f['share_chars_tagged_heading']:.0%}
   of all characters, one heading per *visual line* ({x['heading_marked_over_40_chars']} "headings" are body
   sentences longer than 40 characters).
2. Line breaks become paragraph breaks: only {x['paragraphs_ending_sentence']} of {x['paragraphs']} paragraphs end
   a Korean sentence, so sentences are cut mid-way ("정보 습득이" / "가능하다.").
3. The heading-aware chunker is working as designed: {c['share_boundaries_forced_by_heading']:.0%} of its chunk
   boundaries are heading changes and it never reaches its size limit. But a heading is stored as chunk *metadata*
   and consecutive headings overwrite each other. Only **{c0.in_chunk_text:.0%}** of the guidebook's Hangul words
   reach chunk text ({c0.in_text_or_heading:.0%} including the heading field that is embedded). Most of the
   document is not retrievable at all.
4. Secondary: table cells (tables off) become one-word headings ("## 20시간"); page numbers ({x['page_number_only']}),
   the running header and chapter tabs repeat as paragraphs; BEL/ZWNJ control characters are present. OCR and
   encoding are *not* causes: no replacement characters, 91 of 94 pages have a text layer, and Hangul is intact.
   The other corpus_v2 documents keep >= 97 % of their words in chunks; only this PDF is affected."""))

## 2 · Extraction-quality examples (page 74: part-time work hours table; page 37: registration changes)

In [ ]:
for p in ("74", "37"):
    a = INV["examples"]["production"][p]; b = INV["examples"]["C3_heading"][p]
    display(Markdown(f"**Page {p} - production text (first 700 characters)**\n```\n{a}\n```\n"
                     f"**Page {p} - C3 re-extraction**\n```\n{b}\n```"))

## 3 · Experimental configurations

Same source, same chunker settings (target 320 / max 420 / overlap 48 tokens), same embedding model. Content retention is measured on Hangul characters relative to the production text; the 2.3 % removed by C2-C4 is page furniture (running header x42, chapter tabs, page numbers).

In [ ]:
cfg = {"C0_existing": "production text unchanged", "C1_merge_short": "text-only repair: furniture + control chars removed, false ## dropped, short fragments merged within a page (~400 chars)",
       "C2_paragraph": "PDF re-extraction: lines of a block re-flowed into one paragraph, furniture removed by position, no headings",
       "C3_heading": "C2 + headings from corrected body size (>=1.4x '#', >=1.15x '##')", "C4_table": "C3 + validated tables as | a | b | rows",
       "P1_prod_fixed_body": "production extract_pdf, body size over all pages (minimal production fix)"}
t = pd.DataFrame([dict(variant=LABEL[v], description=cfg[v], chars=m["chars"], hangul_retained=m["hangul_retained_vs_existing"],
                       paragraphs=m["paragraphs"], heading_paragraphs=m["heading_paragraphs"], page_markers=m["pages_marked"],
                       control_chars=m["control_chars"]) for v, m in VM["variants"].items()])
display(t)
print("C4 tables kept / rejected by the 95 % text-retention check:", VM["c4_tables"]["kept"], "/", VM["c4_tables"]["rejected"])

## 4 · Chunk-length distributions

In [ ]:
SENT = re.compile(r"(다|요|음|함|임|됨|것|[.!?:;)\]>》」』])\s*$")
crow = []
for v, d in CH.items():
    idx = json.loads((RES / f"index_{v}.json").read_text())
    crow.append(dict(variant=LABEL[v], chunks=len(d), median_chars=d.chars.median(), p10=d.chars.quantile(.1), p90=d.chars.quantile(.9),
                     under_100=(d.chars < 100).mean(), under_50=(d.chars < 50).mean(),
                     ends_at_sentence=d.content.astype(str).map(lambda s: bool(SENT.search(s))).mean(),
                     with_page_reference=d.page_start.notna().mean(), with_heading=(d.heading.fillna("") != "").mean(),
                     corpus_chunks=idx["chunks"], index_build_s=idx["build_seconds"]))
CROW = pd.DataFrame(crow); display(CROW.round(3))
fig, axes = plt.subplots(2, 3, figsize=(12.5, 6.2), sharex=True, sharey=True)
bins = np.logspace(0, np.log10(2200), 28)
for ax, (v, d) in zip(axes.flat, CH.items()):
    ax.hist(d.chars.clip(lower=1), bins=bins, color=A.COLORS["B0"], edgecolor=A.SURFACE, linewidth=0.8)
    med = d.chars.median(); ax.axvline(med, color=A.INK, linestyle="--", linewidth=1)
    ax.set_xscale("log"); ax.set_title(f"{LABEL[v]}", fontsize=10)
    ax.text(0.03, 0.92, f"n = {len(d)} | median {med:.0f}\n< 100 chars: {(d.chars < 100).mean():.0%}", transform=ax.transAxes,
            fontsize=8.5, va="top", color=A.INK)
for ax in axes[-1]: ax.set_xlabel("characters per chunk (log scale)")
for ax in axes[:, 0]: ax.set_ylabel("chunks")
fig.suptitle("Guidebook chunk length by configuration", fontweight="bold"); fig.tight_layout()
print(A.save(fig, "13_chunk_length_distribution", FIG))

## 5 · Retrieval comparison

Primary: frozen v0.2 labels (held-out). The frozen labels were written before the guidebook existed, so a variant that retrieves the guidebook for a question it answers is *penalised*. The sensitivity analysis adds the AI-proposed guidebook credits (review worksheet `data/review/label_review_new_documents_corpus_v2.csv`, **not human-reviewed**) and is reported separately.

In [ ]:
METRICS = ("r1", "r5", "rr", "ndcg5")
tabs = []
for lab, df in RET.items():
    for ds in ("heldout_v0.2", "pilot_v0.1"):
        s = A.summarize(df[(df.dataset == ds) & df.method.isin(["B0", "E2", "E2S"])], ["variant", "method"], METRICS)
        tabs.append(s.assign(labels=lab, dataset=ds))
SUM = pd.concat(tabs, ignore_index=True); SUM.to_csv(RES / "summary_by_variant.csv", index=False)
show = SUM[(SUM.dataset == "heldout_v0.2") & (SUM.method == "E2")].copy(); show["variant"] = show.variant.map(LABEL)
display(Markdown("**Held-out v0.2, E2 (production hybrid), both label sets**"))
display(show[["labels", "variant", "n", "r1", "r5", "rr", "rr_lo", "rr_hi", "ndcg5"]].round(3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.6), sharey=True)
order = [v for v in VARIANTS + ["C0_repeat"] if v in RUNS]
for ax, lab in zip(axes, ("frozen", "ai_sensitivity")):
    s = SUM[(SUM.dataset == "heldout_v0.2") & (SUM.labels == lab)]
    for j, m in enumerate(("B0", "E2")):
        g = s[s.method == m].set_index("variant").reindex(order)
        y = np.arange(len(order)) + (j - 0.5) * 0.28
        ax.errorbar(g.rr, y, xerr=[g.rr - g.rr_lo, g.rr_hi - g.rr], fmt="o", color=A.COLORS[m], capsize=3, markersize=7,
                    linewidth=1.5, label={"B0": "B0 dense", "E2": "E2 weighted hybrid"}[m])
    ax.set_yticks(np.arange(len(order)), [LABEL[v] for v in order]); ax.grid(axis="x"); ax.grid(axis="y", visible=False)
    ax.set_xlabel("MRR, held-out v0.2 (95% cluster-bootstrap CI)")
    ax.set_title({"frozen": "Frozen labels (official)", "ai_sensitivity": "+ AI-proposed guidebook labels (not reviewed)"}[lab], fontsize=10.5)
axes[0].invert_yaxis()   # shared y-axis: invert once
h, l = axes[0].get_legend_handles_labels()
fig.legend(h, l, loc="lower center", ncol=2, bbox_to_anchor=(0.5, -0.02), frameon=False)
fig.tight_layout(rect=(0, 0.06, 1, 1)); print(A.save(fig, "13_retrieval_by_variant", FIG))

### Korean and cross-language questions (held-out, E2)

In [ ]:
sub = []
for lab, df in RET.items():
    d = df[(df.dataset == "heldout_v0.2") & (df.method == "E2")]
    for name, g in (("Korean questions", d[d.language == "ko"]), ("cross-language questions", d[d.cross_language]),
                    ("same-language questions", d[~d.cross_language])):
        s = A.summarize(g, ["variant"], ("r1", "rr")).assign(group=name, labels=lab); sub.append(s)
SUB = pd.concat(sub, ignore_index=True)
piv = SUB.assign(variant=SUB.variant.map(LABEL)).pivot_table(index="variant", columns=["labels", "group"], values="rr")
display(piv.reindex([LABEL[v] for v in order]).round(3))
print("n per group:", SUB.groupby("group").n.first().to_dict())

### Retrieval latency

Median local query time (Apple M1, CPU, in-memory store). Values come from the dedicated timing pass on a quiet machine (`--latency-only`, 3 repetitions per question) when available; otherwise from the evaluation run, which shared the machine with the reranking job.

In [ ]:
lat = []
for v in order:
    p = RES / f"latency_{v}.json"
    if p.exists():
        L = json.loads(p.read_text()); lat.append(dict(variant=LABEL[v], source="timing pass", B0_ms=L["B0"]["median_ms"], E2_ms=L["E2"]["median_ms"],
                                                     E2_p90_ms=L["E2"]["p90_ms"]))
    else:
        d = RET["frozen"][RET["frozen"].variant == v]
        lat.append(dict(variant=LABEL[v], source="evaluation run (shared machine)", B0_ms=d[d.method == "B0"].latency_ms.median(),
                        E2_ms=d[d.method == "E2"].latency_ms.median(), E2_p90_ms=d[d.method == "E2"].latency_ms.quantile(.9)))
LAT = pd.DataFrame(lat); display(LAT.round(1))

## 6 · Evidence completeness

* **evidence page hit@5** - for held-out scenarios where the review proposes the guidebook as relevant, does an E2 top-5 passage come from the guidebook page(s) that contain the answer? (page references are preserved in every variant)
* **guidebook passage length in top 5** - fragments cannot carry an answer
* **evidence terms@5** - share of the scenario's (mostly English) evidence terms found in the top-5 passages (all held-out questions).

In [ ]:
rev = pd.read_csv(A.DATA / "review/label_review_new_documents_corpus_v2.csv")
def pages(s):
    out = set()
    for part in str(s).split(","):
        part = part.strip()
        if "-" in part: a, b = part.split("-"); out |= set(range(int(a), int(b) + 1))
        elif part.isdigit(): out.add(int(part))
    return out
EVP = {r.scenario_id: pages(r.evidence_pages) for r in rev.itertuples() if r.document_id == GB and r.ai_proposed_relevance == "relevant"}
ev = []
for v in order:
    for line in (RES / f"passages_{v}.jsonl").open(encoding="utf-8"):
        p = json.loads(line); r = REC[p["question_id"]]
        if QDS[p["question_id"]] != "heldout_v0.2": continue
        top = p["passages"]; gb = [x for x in top if x["document_id"] == GB]
        terms = [t.lower() for t in r.get("evidence_terms") or []]
        joined = " ".join((x["text"] or "").lower() for x in top)
        ev.append(dict(variant=v, question_id=p["question_id"], scenario_id=r["scenario_id"], language=r["language"],
                       guidebook_in_top5=len(gb), gb_median_len=np.median([len(x["text"] or "") for x in gb]) if gb else np.nan,
                       fragment_at_rank1=float(bool(top) and top[0]["document_id"] == GB and len(top[0]["text"] or "") < 50),
                       evidence_terms_at5=(sum(t in joined for t in terms) / len(terms)) if terms else np.nan,
                       page_hit=(float(any(x["page"] in EVP[r["scenario_id"]] for x in gb)) if r["scenario_id"] in EVP else np.nan)))
EV = pd.DataFrame(ev)
evs = EV.groupby("variant").agg(questions_with_guidebook_top5=("guidebook_in_top5", lambda s: (s > 0).mean()),
                                median_guidebook_passage_chars=("gb_median_len", "median"),
                                fragment_rank1=("fragment_at_rank1", "mean"), evidence_terms_at5=("evidence_terms_at5", "mean"),
                                evidence_page_hit_at5=("page_hit", "mean"), n_page_questions=("page_hit", "count")).reindex(order)
evs.index = evs.index.map(LABEL); display(evs.round(3))

## 7 · Error analysis (held-out, E2, frozen labels): questions whose first-relevant rank changes vs C0

In [ ]:
F = RET["frozen"]; e2 = F[(F.dataset == "heldout_v0.2") & (F.method == "E2")].pivot_table(index="question_id", columns="variant", values="rank")
e2 = e2.fillna(99)
chg = []
for v in [x for x in order if x != "C0_existing"]:
    better = (e2[v] < e2["C0_existing"]).sum(); worse = (e2[v] > e2["C0_existing"]).sum()
    chg.append(dict(variant=LABEL[v], improved=int(better), worse=int(worse), unchanged=int(len(e2) - better - worse)))
display(pd.DataFrame(chg))
best = max([v for v in VARIANTS[1:] if v in RUNS], key=lambda v: SUM[(SUM.dataset == "pilot_v0.1") & (SUM.labels == "frozen") & (SUM.method == "E2") & (SUM.variant == v)].rr.iloc[0])
print("best variant on the development set (pilot, E2 MRR, frozen labels):", LABEL[best])
top1 = {}
for v in ("C0_existing", best):
    for line in (RES / f"passages_{v}.jsonl").open(encoding="utf-8"):
        p = json.loads(line); top1[(v, p["question_id"])] = p["passages"][0] if p["passages"] else {}
moved = e2[(e2[best] != e2["C0_existing"])].index[:8]
ex = [dict(question=q, lang=REC[q]["language"], rank_C0=e2.loc[q, "C0_existing"], rank_best=e2.loc[q, best],
           C0_top1=("[GB] " if top1[("C0_existing", q)].get("document_id") == GB else "") + str(top1[("C0_existing", q)].get("text") or "")[:70],
           best_top1=("[GB] " if top1[(best, q)].get("document_id") == GB else "") + str(top1[(best, q)].get("text") or "")[:70]) for q in moved]
display(pd.DataFrame(ex).replace(99, "none"))

## 8 · Statistical analysis

Paired comparisons of each variant against C0 on the same held-out questions (E2, MRR and Recall@1), 95 % cluster-bootstrap intervals over the 20 scenarios, exact McNemar for Recall@1, Holm correction across the five variants (MRR, frozen labels). `C0 rebuilt` measures the noise of rebuilding an identical index (random tie-breaking in the local store).

In [ ]:
cmp = []
for lab, df in RET.items():
    d = df[(df.dataset == "heldout_v0.2") & (df.method == "E2")].assign(method=lambda x: x.variant)
    for v in [x for x in order if x != "C0_existing"]:
        for col in ("rr", "r1"):
            r = A.paired_difference(d, v, "C0_existing", col); r.update(labels=lab, variant=v); cmp.append(r)
CMP = pd.DataFrame(cmp)
prim = CMP[(CMP.labels == "frozen") & (CMP.metric == "rr") & (CMP.variant != "C0_repeat")].copy()
from scipy.stats import norm
prim["p_approx"] = [2 * (1 - norm.cdf(abs(r["diff"]) / max((r["hi"] - r["lo"]) / 3.92, 1e-9))) for _, r in prim.iterrows()]
prim = prim.sort_values("p_approx"); m = len(prim)
prim["holm_threshold"] = [0.05 / (m - i) for i in range(m)]
ok = True; sup = []
for _, r in prim.iterrows():
    ok = ok and r["p_approx"] <= r["holm_threshold"]; sup.append(ok)
prim["supported_after_holm"] = sup
CMP.to_csv(RES / "paired_vs_C0.csv", index=False)
display(prim.assign(variant=prim.variant.map(LABEL))[["variant", "n", "mean_a", "mean_b", "diff", "lo", "hi", "p_approx", "holm_threshold", "supported_after_holm"]].round(4))
display(CMP[CMP.metric == "r1"].assign(variant=CMP.variant.map(LABEL))[["labels", "variant", "diff", "lo", "hi", "a_better", "b_better", "mcnemar_p"]].round(4))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.6))
labs = [v for v in order if v != "C0_existing"]
for j, (lab, color) in enumerate((("frozen", A.COLORS["E2"]), ("ai_sensitivity", A.COLORS["E2S"]))):
    s = CMP[(CMP.labels == lab) & (CMP.metric == "rr")].set_index("variant").reindex(labs)
    y = np.arange(len(labs)) + (j - 0.5) * 0.3
    ax.errorbar(100 * s["diff"], y, xerr=[100 * (s["diff"] - s["lo"]), 100 * (s["hi"] - s["diff"])], fmt="o", color=color,
                capsize=3, markersize=7, linewidth=1.5, label={"frozen": "frozen labels", "ai_sensitivity": "+ AI-proposed labels"}[lab])
    for xv, yv in zip(100 * s["diff"], y): ax.text(xv, yv - 0.17, f"{xv:+.1f}", ha="center", fontsize=8, color=A.INK)
ax.axvline(0, color=A.INK2, linewidth=1); ax.set_yticks(np.arange(len(labs)), [LABEL[v] for v in labs]); ax.invert_yaxis()
ax.set_xlabel("E2 MRR difference vs C0, percentage points (95% CI)"); ax.grid(axis="x"); ax.grid(axis="y", visible=False)
ax.set_title("Held-out v0.2 (n = 100 questions, 20 scenarios)")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.14), ncol=2, frameon=False); fig.tight_layout()
print(A.save(fig, "13_paired_differences", FIG))

## 9 · Conclusions and limitations

In [ ]:
def row(lab, v, ds="heldout_v0.2", m="E2"):
    return SUM[(SUM.labels == lab) & (SUM.variant == v) & (SUM.dataset == ds) & (SUM.method == m)].iloc[0]
def pp(v, lab="frozen", col="rr"):
    r = CMP[(CMP.labels == lab) & (CMP.variant == v) & (CMP.metric == col)].iloc[0]
    return f"{100*r['diff']:+.1f} pp [{100*r['lo']:+.1f}, {100*r['hi']:+.1f}]"
cw = CROW.set_index("variant"); ev_ = evs
lower = [LABEL[r.variant] for r in prim.itertuples() if r.supported_after_holm and r.diff < 0]
higher = [LABEL[r.variant] for r in prim.itertuples() if r.supported_after_holm and r.diff > 0]
sens = CMP[(CMP.labels == "ai_sensitivity") & (CMP.metric == "rr") & (CMP.variant != "C0_repeat")]
sens_txt = ", ".join(f"{LABEL[r.variant].split(' ')[0]} {100*r['diff']:+.1f} pp [{100*r['lo']:+.1f}, {100*r['hi']:+.1f}]" for _, r in sens.iterrows())
fr = SUM[(SUM.labels == "frozen") & (SUM.method == "E2")].set_index(["dataset", "variant"]).rr
ko = SUB[(SUB.group == "Korean questions")].set_index(["labels", "variant"]).rr
noise = pp("C0_repeat") if "C0_repeat" in RUNS else "not measured"
g0, g3 = ev_.loc[LABEL["C0_existing"]], ev_.loc[LABEL["C3_heading"]]
text = f"""### Conclusions

* **Cause (verified).** The fragmentation is an *extraction* defect - the body font size is estimated from the
  first 20 pages ({INV['font']['body_estimate_production_first20']} pt instead of {INV['font']['body_estimate_all_pages']} pt), so
  {INV['font']['share_chars_tagged_heading']:.0%} of the text is tagged `##`, one "heading" per visual line - and the
  heading-aware chunker turns it into **content loss**: consecutive headings overwrite each other, so only
  {cov.iloc[0].in_chunk_text:.0%} of the guidebook's Hangul words reach chunk text ({cov.iloc[0].in_text_or_heading:.0%}
  including the embedded heading field). OCR, encoding and the chunker's size logic are not the cause.
* **Repair works at the data level.** Every variant keeps 97-100 % of the words in chunks; median chunk length
  {cw.loc[LABEL['C0_existing'], 'median_chars']:.0f} -> {cw.loc[LABEL['C2_paragraph'], 'median_chars']:.0f}-{cw.loc[LABEL['C1_merge_short'], 'median_chars']:.0f}
  characters; a guidebook passage reaches the E2 top 5 for {g0.questions_with_guidebook_top5:.0%} -> {g3.questions_with_guidebook_top5:.0%}
  of held-out questions, and the page that holds the answer (reviewed scenarios) is hit in the top 5 for
  {g0.evidence_page_hit_at5:.0%} -> {g3.evidence_page_hit_at5:.0%} (C3).
* **But benchmark scores fall.** With the frozen labels, held-out E2 MRR is {fr[('heldout_v0.2', 'C0_existing')]:.3f} for
  production text and {fr[('heldout_v0.2', 'P1_prod_fixed_body')]:.3f}-{fr[('heldout_v0.2', 'C2_paragraph')]:.3f} for the repaired variants;
  significantly lower after Holm correction: {', '.join(lower) or 'none'}; significantly higher: {', '.join(higher) or 'none'}.
  The development set agrees (pilot E2 MRR {fr[('pilot_v0.1', 'C0_existing')]:.3f} vs {fr[('pilot_v0.1', 'C2_paragraph')]:.3f} for C2).
  The rebuilt identical index changes E2 MRR by {noise}, so these are real effects, not tie noise.
* **Why.** A fully indexed 94-page general guide competes with the specific official pages for almost every
  question. Some of those wins are correct answers the frozen labels do not credit (H03: "TOPIK ... 연간 6회", p. 26;
  H06: the D-10 change, p. 75); others are topical distractors (H09-ko: the part-time hours table ranks first for
  "can I work before my permit is approved?"). With the AI-proposed guidebook credits (not
  human-reviewed) the differences shrink and their intervals include zero: {sens_txt}. Korean-question MRR under those
  labels: C0 {ko[('ai_sensitivity', 'C0_existing')]:.3f}, C2 {ko[('ai_sensitivity', 'C2_paragraph')]:.3f}.
* **Recommendation.** (1) Fix the extraction defect anyway - today most of an approved document is silently
  unsearchable, which is a data-integrity bug, not a tuning choice. Minimal production change: body-size estimate over
  all pages (P1); better: re-flow lines and drop page furniture (C2/C3). Re-extract the guidebook as a *new version*
  through the normal approval flow. (2) Do not expect higher benchmark numbers from it: judge the change only after
  human review of the guidebook labels (v0.3), and pair it with a control for broad documents - the reranker
  (notebook 10/12) or a per-document cap / lower authority weight for the 2021 general guide. Nothing was changed in
  production here.

### Limitations

* Labels: the frozen labels pre-date the guidebook; the sensitivity labels are AI-proposed and **not human-reviewed**
  (worksheet status: pending). Document-level binary relevance; 20 held-out scenarios -> wide intervals.
* Only one document differs between variants; evidence completeness uses page-level matches for 7 reviewed scenarios.
* The guidebook is the 2021 edition; some rules may be outdated even where it is "relevant" for retrieval.
* Variants were designed on this one PDF; P1 is the only variant that is the production extractor itself.
* Pages 4, 5 and 91 are images and remain unextracted (OCR not attempted); latency is local (Apple M1), not Render."""
(REP / "chunk_quality_results.md").write_text("# Korean PDF chunk quality (notebook 13)\n\n" + text + "\n")
display(Markdown(text))